# Gloss-to-Healthcare Text Translation Baseline

## Notebook 06: From Recognized Sign Glosses to Healthcare Sentences

This notebook starts the **language translation stage** of the project.

You have already completed the **sign recognition model comparison stage**:

| Notebook | Model | Stage |
|---|---|---|
| 01 | Random Forest | Traditional sign recognition baseline |
| 02 | HMM | Traditional temporal recognition baseline |
| 03 | LSTM | Deep temporal recognition baseline |
| 04 | ST-GCN | Graph-based spatial-temporal recognition |
| 05 | Transformer Encoder | Attention-based recognition |

Those models answer this question:

```text
Can a model recognize a sign from video?
```

This notebook answers the next question:

```text
After signs are recognized, how do we convert sign glosses into meaningful healthcare text?
```

## Important distinction

The recognition models produce labels or gloss-like outputs such as:

```text
pain where
medicine allergy you
doctor wait please
```

But healthcare workers need normal readable sentences such as:

```text
Where is the pain?
Are you allergic to any medicine?
Please wait for the doctor.
```

So this notebook builds a **gloss-to-healthcare-text translation baseline**.

---

# Models used in this notebook

This notebook uses two levels of modelling:

## 1. Rule-based translation baseline

This is the first model used here.

It uses a dictionary/mapping approach:

```text
KSL-style gloss sequence → healthcare sentence
```

Example:

```text
pain where → Where is the pain?
```

This is simple and explainable.  
It is useful when the dataset is small.

## 2. T5/mT5 preparation stage

This notebook also prepares the data format needed for future language models:

| Model | Why it matters |
|---|---|
| T5 | Good for text-to-text transformation |
| mT5 | Better future option for multilingual text such as English, Kiswahili, and KSL-related glosses |

This notebook does **not train T5/mT5 yet** because training a language model requires more phrase pairs.  
Instead, it prepares the dataset structure needed for future fine-tuning.

---

# Why this notebook matters for the final architecture

Your final project architecture is:

```text
KSL video
→ sign recognition model
→ gloss output
→ language model
→ healthcare text
```

This notebook focuses on this part:

```text
gloss output → healthcare text
```

Later, this will connect to:

```text
healthcare text → KSL sign/avatar output
```

## 1. Import libraries

This notebook uses mostly standard Python libraries.

We also use scikit-learn metrics for simple text comparison.

In [1]:
import json
import re
import random
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

print("Notebook started successfully")

Notebook started successfully


## 2. Configure project folders

The notebook saves:

- phrase dataset
- translation results
- model mapping
- metadata

If running inside Docker, outputs will be saved to:

```text
/workspace/dataset/ksl_project_data/
```

If running locally, outputs will be saved to:

```text
ksl_project_data/
```

In [2]:
PROJECT_DIR = Path("/workspace/dataset/ksl_project_data").resolve()

if not Path("/workspace/dataset").exists():
    PROJECT_DIR = Path("ksl_project_data").resolve()

DATA_DIR = PROJECT_DIR / "language_translation"
MODEL_DIR = PROJECT_DIR / "models"
REPORT_DIR = PROJECT_DIR / "reports"

for folder in [PROJECT_DIR, DATA_DIR, MODEL_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Language data directory:", DATA_DIR)
print("Model directory:", MODEL_DIR)
print("Report directory:", REPORT_DIR)

Project directory: /workspace/dataset/ksl_project_data
Language data directory: /workspace/dataset/ksl_project_data/language_translation
Model directory: /workspace/dataset/ksl_project_data/models
Report directory: /workspace/dataset/ksl_project_data/reports


## 3. Define the healthcare phrase dataset

This is a small manually created phrase set for the baseline.

Each row contains:

| Column | Meaning |
|---|---|
| `id` | Phrase number |
| `domain` | Healthcare area such as triage, medicine, emergency |
| `gloss` | Simplified sign/gloss-style input |
| `healthcare_text` | Natural healthcare sentence |
| `priority` | Whether the phrase is normal, important, or critical |

## Why glosses are used

A gloss is a simplified word-based representation of signs.

Example:

```text
you pain where
```

This does not follow normal English grammar, but it represents the sign meaning.

The translation model should convert it to:

```text
Where is your pain?
```

## Note about KSL

These are **starter gloss examples**, not final validated KSL grammar.

For thesis validation, they should later be reviewed by:

- KSL interpreter
- deaf user/community representative
- healthcare worker
- ICT/AI expert

In [3]:
healthcare_phrases = [
    {
        "id": 1,
        "domain": "triage",
        "gloss": "pain where",
        "healthcare_text": "Where is the pain?",
        "priority": "critical",
    },
    {
        "id": 2,
        "domain": "emergency",
        "gloss": "you chest pain have",
        "healthcare_text": "Do you have chest pain?",
        "priority": "critical",
    },
    {
        "id": 3,
        "domain": "medicine",
        "gloss": "medicine allergy you",
        "healthcare_text": "Are you allergic to any medicine?",
        "priority": "critical",
    },
    {
        "id": 4,
        "domain": "pharmacy",
        "gloss": "medicine take day two",
        "healthcare_text": "Take this medicine twice a day.",
        "priority": "important",
    },
    {
        "id": 5,
        "domain": "follow_up",
        "gloss": "come back week one",
        "healthcare_text": "Come back after one week.",
        "priority": "normal",
    },
    {
        "id": 6,
        "domain": "reception",
        "gloss": "doctor wait please",
        "healthcare_text": "Please wait for the doctor.",
        "priority": "normal",
    },
    {
        "id": 7,
        "domain": "emergency",
        "gloss": "breathing difficult you",
        "healthcare_text": "Do you have difficulty breathing?",
        "priority": "critical",
    },
    {
        "id": 8,
        "domain": "assessment",
        "gloss": "pregnant you",
        "healthcare_text": "Are you pregnant?",
        "priority": "important",
    },
    {
        "id": 9,
        "domain": "medicine",
        "gloss": "medicine today take you",
        "healthcare_text": "Have you taken medicine today?",
        "priority": "important",
    },
    {
        "id": 10,
        "domain": "diagnosis",
        "gloss": "hurt where show me",
        "healthcare_text": "Show me where it hurts.",
        "priority": "critical",
    },
    {
        "id": 11,
        "domain": "diagnosis",
        "gloss": "fever you have",
        "healthcare_text": "Do you have a fever?",
        "priority": "important",
    },
    {
        "id": 12,
        "domain": "diagnosis",
        "gloss": "head pain you",
        "healthcare_text": "Do you have a headache?",
        "priority": "normal",
    },
    {
        "id": 13,
        "domain": "emergency",
        "gloss": "blood pressure check need",
        "healthcare_text": "We need to check your blood pressure.",
        "priority": "important",
    },
    {
        "id": 14,
        "domain": "emergency",
        "gloss": "ambulance call need",
        "healthcare_text": "We need to call an ambulance.",
        "priority": "critical",
    },
    {
        "id": 15,
        "domain": "pharmacy",
        "gloss": "medicine after food take",
        "healthcare_text": "Take the medicine after food.",
        "priority": "important",
    },
    {
        "id": 16,
        "domain": "pharmacy",
        "gloss": "medicine before food take",
        "healthcare_text": "Take the medicine before food.",
        "priority": "important",
    },
    {
        "id": 17,
        "domain": "reception",
        "gloss": "name your what",
        "healthcare_text": "What is your name?",
        "priority": "normal",
    },
    {
        "id": 18,
        "domain": "reception",
        "gloss": "age your what",
        "healthcare_text": "What is your age?",
        "priority": "normal",
    },
    {
        "id": 19,
        "domain": "assessment",
        "gloss": "diabetes you have",
        "healthcare_text": "Do you have diabetes?",
        "priority": "important",
    },
    {
        "id": 20,
        "domain": "assessment",
        "gloss": "hospital admit need",
        "healthcare_text": "You need to be admitted to the hospital.",
        "priority": "critical",
    },
]

df = pd.DataFrame(healthcare_phrases)
df

,id,domain,gloss,healthcare_text,priority
0,1,triage,pain where,Where is the pain?,critical
1,2,emergency,you chest pain have,Do you have chest pain?,critical
2,3,medicine,medicine allergy you,Are you allergic to any medicine?,critical
3,4,pharmacy,medicine take day two,Take this medicine twice a day.,important
4,5,follow_up,come back week one,Come back after one week.,normal
5,6,reception,doctor wait please,Please wait for the doctor.,normal
6,7,emergency,breathing difficult you,Do you have difficulty breathing?,critical
7,8,assessment,pregnant you,Are you pregnant?,important
8,9,medicine,medicine today take you,Have you taken medicine today?,important
9,10,diagnosis,hurt where show me,Show me where it hurts.,critical


## 4. Save the starter phrase dataset

This dataset becomes a reusable project artifact.

It can later be expanded with validated KSL glosses and interpreter-reviewed phrase mappings.

In [4]:
phrase_dataset_path = DATA_DIR / "healthcare_gloss_phrase_dataset.csv"
df.to_csv(phrase_dataset_path, index=False)

print("Saved phrase dataset:", phrase_dataset_path)

Saved phrase dataset: /workspace/dataset/ksl_project_data/language_translation/healthcare_gloss_phrase_dataset.csv


## 5. Normalize gloss text

Recognition model output can be noisy.

For example, the model may output:

```text
Pain Where
pain   where
where pain
```

This function cleans the text by:

1. Lowercasing
2. Removing extra spaces
3. Removing punctuation
4. Sorting or preserving tokens depending on strategy

For this baseline, we preserve token order first.

In [5]:
def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text


df["normalized_gloss"] = df["gloss"].apply(normalize_text)
df[["gloss", "normalized_gloss", "healthcare_text"]].head()

,gloss,normalized_gloss,healthcare_text
0,pain where,pain where,Where is the pain?
1,you chest pain have,you chest pain have,Do you have chest pain?
2,medicine allergy you,medicine allergy you,Are you allergic to any medicine?
3,medicine take day two,medicine take day two,Take this medicine twice a day.
4,come back week one,come back week one,Come back after one week.


## 6. Model 1: Exact rule-based gloss translator

This is the first baseline model.

It uses exact dictionary matching:

```text
normalized gloss → healthcare text
```

Example:

```text
pain where → Where is the pain?
```

## Strength

- Very accurate when gloss input exactly matches the dictionary.
- Easy to explain.
- Good for controlled healthcare phrase systems.

## Weakness

- Fails when word order changes.
- Fails when recognition output has missing words.
- Fails when the phrase is not in the dictionary.

In [6]:
class ExactRuleBasedTranslator:
    def __init__(self, phrase_dataframe):
        self.mapping = {}

        for _, row in phrase_dataframe.iterrows():
            normalized_gloss = normalize_text(row["gloss"])
            self.mapping[normalized_gloss] = row["healthcare_text"]

    def translate(self, gloss):
        normalized_gloss = normalize_text(gloss)
        return self.mapping.get(normalized_gloss, "UNKNOWN_PHRASE")


exact_translator = ExactRuleBasedTranslator(df)

test_glosses = [
    "pain where",
    "medicine allergy you",
    "doctor wait please",
    "unknown sign sequence",
]

for gloss in test_glosses:
    print("Gloss:", gloss)
    print("Translation:", exact_translator.translate(gloss))
    print("-" * 60)

Gloss: pain where
Translation: Where is the pain?
------------------------------------------------------------
Gloss: medicine allergy you
Translation: Are you allergic to any medicine?
------------------------------------------------------------
Gloss: doctor wait please
Translation: Please wait for the doctor.
------------------------------------------------------------
Gloss: unknown sign sequence
Translation: UNKNOWN_PHRASE
------------------------------------------------------------


## 7. Model 2: Token-overlap fuzzy translator

Exact matching is too strict.

A recognition model may output the right words in a different order.

Example:

```text
Expected gloss: medicine allergy you
Model output: you medicine allergy
```

The meaning is close, but exact matching would fail.

This second model uses token overlap similarity.

It compares words in the input gloss with each stored phrase and chooses the closest match.

In [7]:
def token_set(text):
    return set(normalize_text(text).split())


def jaccard_similarity(text_a, text_b):
    set_a = token_set(text_a)
    set_b = token_set(text_b)

    if not set_a and not set_b:
        return 1.0

    if not set_a or not set_b:
        return 0.0

    return len(set_a.intersection(set_b)) / len(set_a.union(set_b))


class FuzzyTokenTranslator:
    def __init__(self, phrase_dataframe, threshold=0.5):
        self.phrase_dataframe = phrase_dataframe.copy()
        self.threshold = threshold

    def translate(self, gloss):
        best_score = -1
        best_row = None

        for _, row in self.phrase_dataframe.iterrows():
            score = jaccard_similarity(gloss, row["gloss"])

            if score > best_score:
                best_score = score
                best_row = row

        if best_score < self.threshold:
            return {
                "translation": "UNKNOWN_PHRASE",
                "matched_gloss": None,
                "score": best_score,
                "priority": None,
                "domain": None,
            }

        return {
            "translation": best_row["healthcare_text"],
            "matched_gloss": best_row["gloss"],
            "score": best_score,
            "priority": best_row["priority"],
            "domain": best_row["domain"],
        }


fuzzy_translator = FuzzyTokenTranslator(df, threshold=0.45)

test_glosses = [
    "where pain",
    "you allergy medicine",
    "doctor please wait",
    "breathing you difficult",
    "medicine food after take",
    "random unrelated words",
]

for gloss in test_glosses:
    result = fuzzy_translator.translate(gloss)

    print("Input gloss:", gloss)
    print("Matched gloss:", result["matched_gloss"])
    print("Translation:", result["translation"])
    print("Score:", round(result["score"], 3))
    print("Priority:", result["priority"])
    print("Domain:", result["domain"])
    print("-" * 80)

Input gloss: where pain
Matched gloss: pain where
Translation: Where is the pain?
Score: 1.0
Priority: critical
Domain: triage
--------------------------------------------------------------------------------
Input gloss: you allergy medicine
Matched gloss: medicine allergy you
Translation: Are you allergic to any medicine?
Score: 1.0
Priority: critical
Domain: medicine
--------------------------------------------------------------------------------
Input gloss: doctor please wait
Matched gloss: doctor wait please
Translation: Please wait for the doctor.
Score: 1.0
Priority: normal
Domain: reception
--------------------------------------------------------------------------------
Input gloss: breathing you difficult
Matched gloss: breathing difficult you
Translation: Do you have difficulty breathing?
Score: 1.0
Priority: critical
Domain: emergency
--------------------------------------------------------------------------------
Input gloss: medicine food after take
Matched gloss: medicine

## 8. Simulate recognition model outputs

In the real system, the input to this notebook would come from models such as:

```text
Transformer recognition model
ST-GCN recognition model
LSTM recognition model
```

But for this baseline, we simulate recognized gloss outputs.

Some examples are exact.  
Some have word order changes.  
Some have missing words.  
Some are wrong.

In [8]:
simulated_recognition_outputs = [
    {
        "expected_gloss": "pain where",
        "recognized_gloss": "where pain",
    },
    {
        "expected_gloss": "medicine allergy you",
        "recognized_gloss": "you medicine allergy",
    },
    {
        "expected_gloss": "doctor wait please",
        "recognized_gloss": "doctor please wait",
    },
    {
        "expected_gloss": "breathing difficult you",
        "recognized_gloss": "breathing you difficult",
    },
    {
        "expected_gloss": "medicine after food take",
        "recognized_gloss": "medicine food after take",
    },
    {
        "expected_gloss": "come back week one",
        "recognized_gloss": "come week",
    },
    {
        "expected_gloss": "fever you have",
        "recognized_gloss": "you fever",
    },
    {
        "expected_gloss": "blood pressure check need",
        "recognized_gloss": "blood pressure need",
    },
    {
        "expected_gloss": "ambulance call need",
        "recognized_gloss": "ambulance need",
    },
    {
        "expected_gloss": "name your what",
        "recognized_gloss": "your name what",
    },
]

sim_df = pd.DataFrame(simulated_recognition_outputs)

# Add expected healthcare sentence from the dataset
expected_map = dict(zip(df["normalized_gloss"], df["healthcare_text"]))

sim_df["expected_text"] = sim_df["expected_gloss"].apply(
    lambda x: expected_map.get(normalize_text(x), "UNKNOWN")
)

sim_df

,expected_gloss,recognized_gloss,expected_text
0,pain where,where pain,Where is the pain?
1,medicine allergy you,you medicine allergy,Are you allergic to any medicine?
2,doctor wait please,doctor please wait,Please wait for the doctor.
3,breathing difficult you,breathing you difficult,Do you have difficulty breathing?
4,medicine after food take,medicine food after take,Take the medicine after food.
5,come back week one,come week,Come back after one week.
6,fever you have,you fever,Do you have a fever?
7,blood pressure check need,blood pressure need,We need to check your blood pressure.
8,ambulance call need,ambulance need,We need to call an ambulance.
9,name your what,your name what,What is your name?


## 9. Evaluate exact translator vs fuzzy translator

We compare:

| Model | Description |
|---|---|
| Exact rule-based translator | Requires exact gloss match |
| Fuzzy token translator | Allows word order variation and partial matches |

This shows why fuzzy matching is more robust when recognition output is imperfect.

In [9]:
exact_predictions = []
fuzzy_predictions = []
fuzzy_scores = []
matched_glosses = []

for _, row in sim_df.iterrows():
    recognized_gloss = row["recognized_gloss"]

    exact_prediction = exact_translator.translate(recognized_gloss)
    fuzzy_result = fuzzy_translator.translate(recognized_gloss)

    exact_predictions.append(exact_prediction)
    fuzzy_predictions.append(fuzzy_result["translation"])
    fuzzy_scores.append(fuzzy_result["score"])
    matched_glosses.append(fuzzy_result["matched_gloss"])

sim_df["exact_prediction"] = exact_predictions
sim_df["fuzzy_prediction"] = fuzzy_predictions
sim_df["fuzzy_score"] = fuzzy_scores
sim_df["matched_gloss"] = matched_glosses

sim_df["exact_correct"] = sim_df["exact_prediction"] == sim_df["expected_text"]
sim_df["fuzzy_correct"] = sim_df["fuzzy_prediction"] == sim_df["expected_text"]

exact_accuracy = sim_df["exact_correct"].mean()
fuzzy_accuracy = sim_df["fuzzy_correct"].mean()

print("Exact translator accuracy:", round(exact_accuracy, 4))
print("Fuzzy translator accuracy:", round(fuzzy_accuracy, 4))

sim_df

Exact translator accuracy: 0.0
Fuzzy translator accuracy: 1.0


,expected_gloss,recognized_gloss,expected_text,exact_prediction,fuzzy_prediction,fuzzy_score,matched_gloss,exact_correct,fuzzy_correct
0,pain where,where pain,Where is the pain?,UNKNOWN_PHRASE,Where is the pain?,1.000000,pain where,False,True
1,medicine allergy you,you medicine allergy,Are you allergic to any medicine?,UNKNOWN_PHRASE,Are you allergic to any medicine?,1.000000,medicine allergy you,False,True
2,doctor wait please,doctor please wait,Please wait for the doctor.,UNKNOWN_PHRASE,Please wait for the doctor.,1.000000,doctor wait please,False,True
3,breathing difficult you,breathing you difficult,Do you have difficulty breathing?,UNKNOWN_PHRASE,Do you have difficulty breathing?,1.000000,breathing difficult you,False,True
4,medicine after food take,medicine food after take,Take the medicine after food.,UNKNOWN_PHRASE,Take the medicine after food.,1.000000,medicine after food take,False,True
5,come back week one,come week,Come back after one week.,UNKNOWN_PHRASE,Come back after one week.,0.500000,come back week one,False,True
6,fever you have,you fever,Do you have a fever?,UNKNOWN_PHRASE,Do you have a fever?,0.666667,fever you have,False,True
7,blood pressure check need,blood pressure need,We need to check your blood pressure.,UNKNOWN_PHRASE,We need to check your blood pressure.,0.750000,blood pressure check need,False,True
8,ambulance call need,ambulance need,We need to call an ambulance.,UNKNOWN_PHRASE,We need to call an ambulance.,0.666667,ambulance call need,False,True
9,name your what,your name what,What is your name?,UNKNOWN_PHRASE,What is your name?,1.000000,name your what,False,True


## 10. Save translation evaluation results

The output table records:

- expected gloss
- simulated recognized gloss
- expected healthcare text
- exact translation prediction
- fuzzy translation prediction
- fuzzy match score
- whether each method was correct

In [10]:
translation_results_path = REPORT_DIR / "gloss_to_healthcare_translation_results.csv"
sim_df.to_csv(translation_results_path, index=False)

print("Saved translation results:", translation_results_path)

Saved translation results: /workspace/dataset/ksl_project_data/reports/gloss_to_healthcare_translation_results.csv


## 11. Prepare dataset for future T5/mT5 fine-tuning

T5 and mT5 are text-to-text transformer models.

They can be trained using examples like:

```text
input: translate gloss to healthcare text: medicine allergy you
target: Are you allergic to any medicine?
```

This notebook prepares the correct format, but does not train T5/mT5 yet.

## Why not train T5/mT5 immediately?

The current phrase set is too small.

Training a language model properly requires more phrase pairs and validation data.

For now, this step prepares the dataset structure.

In [11]:
t5_training_rows = []

for _, row in df.iterrows():
    t5_training_rows.append(
        {
            "input_text": f"translate gloss to healthcare text: {row['gloss']}",
            "target_text": row["healthcare_text"],
            "domain": row["domain"],
            "priority": row["priority"],
        }
    )

t5_df = pd.DataFrame(t5_training_rows)

t5_dataset_path = DATA_DIR / "t5_gloss_to_healthcare_text_dataset.csv"
t5_df.to_csv(t5_dataset_path, index=False)

print("Saved T5/mT5 training-format dataset:", t5_dataset_path)

t5_df.head()

Saved T5/mT5 training-format dataset: /workspace/dataset/ksl_project_data/language_translation/t5_gloss_to_healthcare_text_dataset.csv


,input_text,target_text,domain,priority
0,translate gloss to healthcare text: pain where,Where is the pain?,triage,critical
1,translate gloss to healthcare text: you chest ...,Do you have chest pain?,emergency,critical
2,translate gloss to healthcare text: medicine a...,Are you allergic to any medicine?,medicine,critical
3,translate gloss to healthcare text: medicine t...,Take this medicine twice a day.,pharmacy,important
4,translate gloss to healthcare text: come back ...,Come back after one week.,follow_up,normal


## 12. Create train/test split for future language model training

This creates a small split for future experiments.

In a real project, you should expand the dataset first.

In [13]:
train_df, test_df = train_test_split(
    t5_df,
    test_size=0.25,
    random_state=42,
    stratify=t5_df["priority"],
)

train_path = DATA_DIR / "t5_train.csv"
test_path = DATA_DIR / "t5_test.csv"

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Saved train file:", train_path)
print("Saved test file:", test_path)

Train rows: 15
Test rows: 5
Saved train file: /workspace/dataset/ksl_project_data/language_translation/t5_train.csv
Saved test file: /workspace/dataset/ksl_project_data/language_translation/t5_test.csv


## 13. Optional: T5/mT5 model plan

This cell does not train the model.

It documents the exact future model options.

## Future model options

| Model | Use |
|---|---|
| T5-small | English gloss-to-text baseline |
| mT5-small | Multilingual gloss-to-text baseline |
| ByT5 | Robust text model that works at byte/character level |

## Recommended next language model

For your project, the recommended future model is:

```text
mT5-small
```

because your final KSL healthcare system may involve:

- English
- Kiswahili
- KSL glosses
- possibly Sheng/local expressions

In [14]:
future_model_plan = {
    "current_notebook_model_1": "Exact rule-based translator",
    "current_notebook_model_2": "Fuzzy token-overlap translator",
    "future_model_1": "T5-small",
    "future_model_2": "mT5-small",
    "future_model_3": "ByT5-small",
    "recommended_future_model": "mT5-small",
    "reason": "mT5 is more suitable for multilingual gloss-to-text transformation involving English, Kiswahili, and KSL-related phrase structures.",
}

future_model_plan_path = DATA_DIR / "future_language_model_plan.json"
future_model_plan_path.write_text(json.dumps(future_model_plan, indent=2), encoding="utf-8")

print("Saved future model plan:", future_model_plan_path)
future_model_plan

Saved future model plan: /workspace/dataset/ksl_project_data/language_translation/future_language_model_plan.json


{'current_notebook_model_1': 'Exact rule-based translator',
 'current_notebook_model_2': 'Fuzzy token-overlap translator',
 'future_model_1': 'T5-small',
 'future_model_2': 'mT5-small',
 'future_model_3': 'ByT5-small',
 'recommended_future_model': 'mT5-small',
 'reason': 'mT5 is more suitable for multilingual gloss-to-text transformation involving English, Kiswahili, and KSL-related phrase structures.'}

## 14. Save the rule-based translator model

The exact and fuzzy translators are based on the same phrase mapping.

We save the phrase mapping as JSON so that it can be loaded later by another notebook or application.

In [15]:
phrase_mapping = {}

for _, row in df.iterrows():
    phrase_mapping[normalize_text(row["gloss"])] = {
        "healthcare_text": row["healthcare_text"],
        "domain": row["domain"],
        "priority": row["priority"],
    }

mapping_path = MODEL_DIR / "gloss_to_healthcare_phrase_mapping.json"
mapping_path.write_text(json.dumps(phrase_mapping, indent=2), encoding="utf-8")

print("Saved phrase mapping model:", mapping_path)

Saved phrase mapping model: /workspace/dataset/ksl_project_data/models/gloss_to_healthcare_phrase_mapping.json


## 15. Create metadata summary

This metadata describes:

- what models were used
- what they do
- what files were generated
- how this notebook fits the project architecture

In [17]:
metadata = {
    "notebook": "06_gloss_to_healthcare_text_translation_baseline.ipynb",
    "stage": "Language translation baseline",
    "input": "Recognized sign gloss sequence",
    "output": "Natural healthcare sentence",
    "models_used": [
        {
            "name": "ExactRuleBasedTranslator",
            "type": "rule-based dictionary model",
            "purpose": "Maps exact normalized glosses to healthcare sentences",
        },
        {
            "name": "FuzzyTokenTranslator",
            "type": "token-overlap similarity model",
            "purpose": "Handles reordered or partially recognized glosses",
        },
        {
            "name": "T5/mT5 preparation",
            "type": "future transformer language model preparation",
            "purpose": "Prepares input-target pairs for future gloss-to-text fine-tuning",
        },
    ],
    "exact_translator_accuracy_on_simulated_outputs": float(exact_accuracy),
    "fuzzy_translator_accuracy_on_simulated_outputs": float(fuzzy_accuracy),
    "number_of_phrase_pairs": int(len(df)),
    "generated_files": {
        "phrase_dataset": str(phrase_dataset_path),
        "translation_results": str(translation_results_path),
        "t5_dataset": str(t5_dataset_path),
        "t5_train": str(train_path),
        "t5_test": str(test_path),
        "phrase_mapping": str(mapping_path),
        "future_model_plan": str(future_model_plan_path),
    },
    "created_at": datetime.now().isoformat(),
}

metadata_path = REPORT_DIR / "gloss_to_healthcare_translation_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Saved metadata:", metadata_path)
metadata

Saved metadata: /workspace/dataset/ksl_project_data/reports/gloss_to_healthcare_translation_metadata.json


{'notebook': '06_gloss_to_healthcare_text_translation_baseline.ipynb',
 'stage': 'Language translation baseline',
 'input': 'Recognized sign gloss sequence',
 'output': 'Natural healthcare sentence',
 'models_used': [{'name': 'ExactRuleBasedTranslator',
   'type': 'rule-based dictionary model',
   'purpose': 'Maps exact normalized glosses to healthcare sentences'},
  {'name': 'FuzzyTokenTranslator',
   'type': 'token-overlap similarity model',
   'purpose': 'Handles reordered or partially recognized glosses'},
  {'name': 'T5/mT5 preparation',
   'type': 'future transformer language model preparation',
   'purpose': 'Prepares input-target pairs for future gloss-to-text fine-tuning'}],
 'exact_translator_accuracy_on_simulated_outputs': 0.0,
 'fuzzy_translator_accuracy_on_simulated_outputs': 1.0,
 'number_of_phrase_pairs': 20,
 'generated_files': {'phrase_dataset': '/workspace/dataset/ksl_project_data/language_translation/healthcare_gloss_phrase_dataset.csv',
  'translation_results': '/wo

## 16. How to interpret this notebook in the thesis

You can use this explanation:

> After evaluating sign recognition models, a gloss-to-healthcare-text translation baseline was implemented to demonstrate how recognized sign outputs can be converted into meaningful healthcare sentences. Two baseline translation approaches were tested: an exact rule-based translator and a fuzzy token-overlap translator. The exact translator performs well when recognized glosses match stored phrase patterns exactly, while the fuzzy translator is more robust to word order changes and partial recognition errors. The notebook also prepares training-format data for future T5 or mT5 fine-tuning, which would allow more flexible gloss-to-text translation once a larger validated KSL healthcare phrase dataset is available.

## What has been achieved

This notebook completes the first language stage:

```text
recognized gloss → healthcare sentence
```

## What comes next

The next notebook should be:

```text
07_healthcare_phrase_validation_matrix.ipynb
```

That notebook will prepare a validation matrix for:

- KSL interpreter review
- deaf user review
- healthcare worker review
- technical feasibility review

After validation, the project can move to:

```text
08_text_to_sign_generation_design.ipynb
```

which will explain the diffusion/avatar generation side.